<a href="https://colab.research.google.com/github/arauch6363-crypto/ptnew/blob/main/PT_Predictor_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏇 French Horse Racing Predictor

End-to-end pipeline: data loading → feature engineering → LightGBM model → evaluation.

In [31]:
# ── Standard Library ──────────────────────────────────────────────────────────
import sys, time, ast, json, re
from datetime import datetime, timedelta
from itertools import combinations

# ── Data ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from tqdm import tqdm

# ── Stats ─────────────────────────────────────────────────────────────────────
from scipy.stats import ttest_ind

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, f1_score, classification_report,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import IsolationForest

# ── Boosting ──────────────────────────────────────────────────────────────────
import lightgbm as lgb

# ── Optuna (optional — install if missing) ────────────────────────────────────
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _OPTUNA_AVAILABLE = True
except ImportError:
    _OPTUNA_AVAILABLE = False

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt


## 1  Configuration

In [32]:
# ── Mode: 'train' | 'test' | 'live' ──────────────────────────────────────────
MODE  = 'train'
TODAY = '2026-05-27'

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_PATH = '/content/drive/MyDrive/PT'

PATHS = {
    'runners':    f'{BASE_PATH}/runners.parquet',
    'webTips':    f'{BASE_PATH}/webTips.parquet',
    'odds':       f'{BASE_PATH}/odds.parquet',
    'races':      f'{BASE_PATH}/races.parquet',
    'runners_tdy': f'{BASE_PATH}/runners_tdy.parquet',
    'webTips_tdy': f'{BASE_PATH}/webTips_tdy.parquet',
    'odds_tdy':    f'{BASE_PATH}/odds_tdy.parquet',
    'races_tdy':   f'{BASE_PATH}/races_tdy.parquet',
    'tips_out':    f'{BASE_PATH}/today_tips.parquet',
}

# ── Model training window ─────────────────────────────────────────────────────
TRAIN_START = '2022-06-01'
TRAIN_END   = '2025-10-01'

# ── Market filter ─────────────────────────────────────────────────────────────
ODDS_SUM_MIN, ODDS_SUM_MAX = 1.1, 1.4

# ── Place-market overround ────────────────────────────────────────────────────
PLACE_MARGIN  = 1.2
WIN_OVERROUND = 1.175

# ── Optuna / win-model flags ──────────────────────────────────────────────────
OPTUNA_ENABLED  = True
TRAIN_WIN_MODEL = True

# ── EV threshold ──────────────────────────────────────────────────────────────
EV_MIN = 1.05

# ── Betting odds window ───────────────────────────────────────────────────────
BET_ODDS_MIN = 3.0    # None = no lower bound
BET_ODDS_MAX = 6.0    # None = no upper bound

# ── Market rank filter for top pick ──────────────────────────────────────────
# Diagnostics: edge lives in 50–75% zone (model picks a near-outsider the market
# undervalues).  0 = favourite end, 1 = outsider end.  Set None to disable.
BET_MRP_MIN = None    # None = disabled (start wide; narrow once confirmed OOS)
BET_MRP_MAX = None

# ── Default F1 threshold ──────────────────────────────────────────────────────
best_thresh = 0.5

print(f'Mode: {MODE}  |  Today: {TODAY}  |  EV_MIN: {EV_MIN}')
print(f'Odds window:    {BET_ODDS_MIN}–{BET_ODDS_MAX}')
print(f'MRP filter:     {BET_MRP_MIN}–{BET_MRP_MAX}  (0=fav end, 1=outsider end; None=off)')

Mode: train  |  Today: 2026-05-27  |  EV_MIN: 1.05
Odds window:    3.0–6.0
MRP filter:     None–None  (0=fav end, 1=outsider end; None=off)


## 2  Mount Google Drive (Colab only)

In [33]:
#Uncomment when running in Google Colab
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3  Load Data

In [34]:
def load_parquet(path, dedup_cols=None):
    df = pd.read_parquet(path)
    if dedup_cols:
        df = df.drop_duplicates(subset=dedup_cols)
    else:
        df = df.drop_duplicates()
    return df

# ── Historical data ───────────────────────────────────────────────────────────
runners  = load_parquet(PATHS['runners'],  dedup_cols=['horseName', 'raceId'])
webTips  = load_parquet(PATHS['webTips'],  dedup_cols=['meetingId', 'raceId'])
webTips  = webTips[webTips['text'].notna()]
odds     = load_parquet(PATHS['odds'])
races    = load_parquet(PATHS['races'])

# ── Today's data ──────────────────────────────────────────────────────────────
runners_tdy  = load_parquet(PATHS['runners_tdy'],  dedup_cols=['horseName', 'raceId'])
webTips_tdy  = load_parquet(PATHS['webTips_tdy'],  dedup_cols=['meetingId', 'raceId'])
webTips_tdy  = webTips_tdy[webTips_tdy['text'].notna()]
odds_tdy     = load_parquet(PATHS['odds_tdy'])
races_tdy    = load_parquet(PATHS['races_tdy'])

print(f'Runners: {runners.shape}  |  Races: {races.shape}  |  Odds: {odds.shape}')
print(f'Today runners: {runners_tdy.shape}  |  Today races: {races_tdy.shape}')


Runners: (317172, 59)  |  Races: (28936, 29)  |  Odds: (288231, 13)
Today runners: (73, 59)  |  Today races: (8, 30)


## 4  Merge DataFrames

In [35]:
RACE_COLS = ['id_race', 'class', 'type', 'going', 'date', 'totalPrize',
             'distance', 'name_meeting', 'direction', 'rail', 'surface']

def merge_all(runners_df, races_df, odds_df, webTips_df):
    df = runners_df.merge(races_df[RACE_COLS],
                          left_on='raceId', right_on='id_race', how='left')
    df = df.merge(odds_df[['raceId', 'horseName', 'referenceOdd', 'liveOdd']],
                  on=['raceId', 'horseName'], how='left')
    df = df.merge(webTips_df[['raceId', 'meetingId', 'text']],
                  on=['raceId', 'meetingId'], how='left')
    return df

runners     = merge_all(runners,     races,     odds,     webTips)
runners_tdy = merge_all(runners_tdy, races_tdy, odds_tdy, webTips_tdy)

print(f'Merged runners: {runners.shape}  |  Today: {runners_tdy.shape}')
runners = runners[runners['ranking'].isnull()==False]

Merged runners: (317188, 73)  |  Today: (73, 73)


In [36]:
runners = runners.sort_values(["horseId", "date"])  # wichtig für korrekte Reihenfolge

runners["horse_run"] = runners.groupby("horseId").cumcount() + 1

## 5  Combine Historical + Today (live mode only)

In [37]:
if MODE == 'live':
    runners = pd.concat([runners, runners_tdy], ignore_index=True)
    print(f'Combined runners: {runners.shape}')


## 6  Feature Engineering

### 6.1  Margin → Lengths Back

In [38]:
# ── Margin string/number → numeric lengths ────────────────────────────────────
MARGIN_MAP = {
    'DH': 0, 'NEZ': 0.05, 'CTT': 0.1,
    'TETE': 0.2, 'CTE': 0.3, 'ENC': 0.4, 'LOIN': 99
}

def parse_margin(margin) -> float:
    if isinstance(margin, str):
        m = margin.replace('(', '').replace(')', '').strip()
        if m in MARGIN_MAP:
            return MARGIN_MAP[m]
        total = 0.0
        for part in m.split():
            if '/' in part:
                num, denom = part.split('/')
                total += float(num) / float(denom)
            else:
                try:
                    total += float(part)
                except ValueError:
                    pass
        return total
    if pd.isna(margin):
        return 0.0
    return float(margin)

def build_margin_lookup(df: pd.DataFrame) -> dict:
    return {(row.raceId, row.ranking): row.margin for row in df.itertuples(index=False)}

def margin_to_lengths(margin, ranking, raceId, lookup) -> float:
    """Convert margin to lengths-back value (negative = ahead of field)."""
    if ranking == 1:
        margin_r2 = lookup.get((raceId, 2))
        if margin_r2 is None:
            return 0.0
        return -parse_margin(margin_r2)
    return parse_margin(margin)

# ── Apply ─────────────────────────────────────────────────────────────────────
if MODE != 'live':
    runners = runners[runners['ranking'].notna()]

margin_lookup = build_margin_lookup(runners)

runners['lengths_back'] = runners.apply(
    lambda r: margin_to_lengths(r['margin'], r['ranking'], r['raceId'], margin_lookup),
    axis=1
)
runners['cumulative_lengths_back'] = (
    runners[runners['lengths_back'] >= 0]
    .sort_values('ranking')
    .groupby('raceId')['lengths_back']
    .cumsum()
)

runners['cumulative_lengths_back'] = runners['cumulative_lengths_back'].fillna(runners['lengths_back'])
print('lengths_back computed')


lengths_back computed


### 6.2  Position & Place Labels

In [39]:
runners['runners'] = runners.groupby('raceId')['horseId'].transform('count')
runners['pos_perc'] = (runners['runners'] - runners['ranking']) / (runners['runners'] - 1)
runners['place']    = np.where(
    (runners['ranking'] <= 2) | ((runners['ranking'] == 3) & (runners['runners'] >= 7)), 1, 0
)
runners['win'] = (runners['ranking'] == 1).astype(int)



### 6.3  Going Category & Distance Group

In [40]:
GOING_MAP = {
    'Lourd': 'VERY SLOW', 'Très lourd': 'VERY SLOW', 'Collant': 'VERY SLOW',
    'Souple': 'SLOW',     'Très souple': 'SLOW',
    'Bon souple': 'FAST', 'Bon': 'FAST',
    'Léger': 'VERY FAST', 'Bon léger': 'VERY FAST', 'Très léger': 'VERY FAST',
    'PSF Standard': 'PSF', 'PSF Lente': 'PSF', 'PSF Rapide': 'PSF',
}

def distance_group(m) -> str | None:
    if pd.isna(m):
        return None
    m = int(m)
    if m <= 1000:  return '0-1000'
    if m > 3600:   return '>3600'
    lo = ((m - 1) // 200) * 200 + 1
    return f'{lo}-{lo + 199}'

runners['going_category'] = runners['going'].map(GOING_MAP)
runners['distance_group'] = runners['distance'].apply(distance_group)
runners['date']            = pd.to_datetime(runners['date'])

# Convenience alias
if 'meetingName' not in runners.columns and 'name_meeting' in runners.columns:
    runners['meetingName'] = runners['name_meeting']

# Apply same transformations to today's races (manual going overrides allowed)
races_tdy['going_category'] = races_tdy['going'].map(GOING_MAP)
races_tdy['distance_group'] = races_tdy['distance'].apply(distance_group)

print('going_category & distance_group done')
print('going_category values:', runners['going_category'].unique())

going_category & distance_group done
going_category values: ['PSF' 'SLOW' 'VERY SLOW' 'FAST' 'VERY FAST' nan]


In [41]:
# ── Rating engine ─────────────────────────────────────────────────────────────

kg_per_length = {
    'VERY SLOW': {'0-1000': 1.2, '1001-1200': 1.1, '1201-1400': 1.0,
                  '1401-1600': 0.95, '1601-1800': 0.9, '1801-2000': 0.85,
                  '2001-2200': 0.8, '2201-2400': 0.75, '2401-2600': 0.7,
                  '2601-2800': 0.65, '2801-3000': 0.6, '3001-3200': 0.55,
                  '3201-3400': 0.5, '3401-3600': 0.45, '>3600': 0.4},
    'SLOW':      {'0-1000': 1.3, '1001-1200': 1.2, '1201-1400': 1.1,
                  '1401-1600': 1.05, '1601-1800': 1.0, '1801-2000': 0.95,
                  '2001-2200': 0.9, '2201-2400': 0.85, '2401-2600': 0.8,
                  '2601-2800': 0.75, '2801-3000': 0.7, '3001-3200': 0.65,
                  '3201-3400': 0.6, '3401-3600': 0.55, '>3600': 0.5},
    'FAST':      {'0-1000': 1.5, '1001-1200': 1.4, '1201-1400': 1.3,
                  '1401-1600': 1.2, '1601-1800': 1.15, '1801-2000': 1.1,
                  '2001-2200': 1.05, '2201-2400': 1.0, '2401-2600': 0.95,
                  '2601-2800': 0.9, '2801-3000': 0.85, '3001-3200': 0.8,
                  '3201-3400': 0.75, '3401-3600': 0.7, '>3600': 0.65},
    'VERY FAST': {'0-1000': 1.6, '1001-1200': 1.5, '1201-1400': 1.4,
                  '1401-1600': 1.3, '1601-1800': 1.25, '1801-2000': 1.2,
                  '2001-2200': 1.15, '2201-2400': 1.1, '2401-2600': 1.05,
                  '2601-2800': 1.0, '2801-3000': 0.95, '3001-3200': 0.9,
                  '3201-3400': 0.85, '3401-3600': 0.8, '>3600': 0.75},
    'PSF':       {'0-1000': 1.4, '1001-1200': 1.3, '1201-1400': 1.2,
                  '1401-1600': 1.1, '1601-1800': 1.05, '1801-2000': 1.0,
                  '2001-2200': 0.95, '2201-2400': 0.9, '2401-2600': 0.85,
                  '2601-2800': 0.8, '2801-3000': 0.75, '3001-3200': 0.7,
                  '3201-3400': 0.65, '3401-3600': 0.6, '>3600': 0.55},
}

def lengths_to_rating(l):
    if l < 0:
        raise ValueError("Length must be non-negative")
    if l >= 6.0:
        return 5.0
    x = l / 6.0
    y_norm = np.log1p(9 * x) / np.log(10)
    return 1.0 + 4.0 * y_norm

def signed_bucket(l_signed):
    magnitude = lengths_to_rating(abs(l_signed))
    return magnitude if l_signed >= 0 else -magnitude

def update_ratings_with_history(df, k_factor=1.0, alpha=0.07, initial_ratings=None):
    ratings = dict(initial_ratings) if initial_ratings else {}
    rating_history = []
    df = df.sort_values(by='date')
    df['prize_quantile'] = df.groupby('raceId')['totalPrize_y'].rank(pct=True)
    race_ids = df['raceId'].unique()
    for race_id in tqdm(race_ids, desc="Processing races"):
        race = df[df['raceId'] == race_id].copy()
        for _, row in race.iterrows():
            h = row['horseId']
            if h not in ratings:
                if 'handicapRatingKg' in row and not pd.isnull(row['handicapRatingKg']):
                    ratings[h] = 30.0 + (row['handicapRatingKg'] - 30.0)
                else:
                    ratings[h] = 30.0
        horses = {
            row['horseId']: {
                'rating': ratings[row['horseId']],
                'lengths_back': row['cumulative_lengths_back'],
                'weight': row['weightKg']
            }
            for _, row in race.iterrows()
        }
        surprises = {h: 0.0 for h in horses}
        n_duels   = {h: 0   for h in horses}
        horse_ids = list(horses.keys())
        for i in range(len(horse_ids)):
            for j in range(i + 1, len(horse_ids)):
                h1, h2 = horse_ids[i], horse_ids[j]
                d1, d2 = horses[h1], horses[h2]
                perf_diff = (d1['rating'] - 0.625 * d1['weight']) - (d2['rating'] - 0.625 * d2['weight'])
                bucket_exp = signed_bucket(perf_diff)
                going    = race.iloc[0].get('going_category', 'FAST') or 'FAST'
                distance = race.iloc[0].get('distance_group')
                multiplier = kg_per_length.get(going.upper(), kg_per_length['FAST']).get(distance, 1.0)
                length_gap = d2['lengths_back'] - d1['lengths_back']
                actual_diff = length_gap * multiplier
                bucket_act = signed_bucket(actual_diff)
                surprise = bucket_act - bucket_exp
                surprises[h1] += surprise
                surprises[h2] -= surprise
                n_duels[h1] += 1
                n_duels[h2] += 1
        for h in horses:
            avg_surprise = surprises[h] / max(1, n_duels[h])
            horse_row = race[race['horseId'] == h].iloc[0]
            prize_quantile = horse_row.get('prize_quantile', 0.5)
            horse_run      = horse_row.get('horse_run', 1)
            prize_bonus    = (prize_quantile - 0.5) * 5 * np.exp(-alpha * horse_run)
            ratings[h] += k_factor * (avg_surprise + prize_bonus)
            rating_history.append({'raceId': race_id, 'horseId': h, 'rating_after_race': ratings[h]})
    history_df = pd.DataFrame(rating_history)
    return df.merge(history_df, on=['raceId', 'horseId'], how='left')

In [42]:
import pandas as pd
import numpy as np
import gc
from tqdm import tqdm

TOL = 0.075

runners_sorted = runners.sort_values('date')

# Erste bekannte Rating pro Pferd
first_non_null = (
    runners_sorted[runners_sorted['handicapRatingKg'].notna()]
    .groupby('horseId')
    .first()[['handicapRatingKg', 'date']]
    .reset_index()
    .rename(columns={'date': 'first_rating_date'})
)

bins   = [0, 0.2, 0.4, 0.6, 0.8, 1.001]
labels = ['0.0-0.2', '0.2-0.4', '0.4-0.6', '0.6-0.8', '0.8-1.0']
runners['pos_perc_bin'] = pd.cut(runners['pos_perc'], bins=bins, labels=labels, right=False)

SENTINEL = '__NULL__'
for col, sentinel in [('class', SENTINEL), ('type', SENTINEL), ('age', -1)]:
    runners[f'{col}_join'] = runners[col].fillna(sentinel)

# Referenz-Pool: nur Pferde mit bekannter Rating
ref_pool = (
    runners[runners['handicapRatingKg'].notna()][[
        'horseId', 'pos_perc_bin', 'age_join', 'class_join',
        'type_join', 'totalPrize_y', 'date', 'handicapRatingKg'
    ]]
    .merge(first_non_null[['horseId', 'first_rating_date']], on='horseId', how='left')
    .copy()
)
ref_pool['handicapRatingKg'] = ref_pool['handicapRatingKg'].astype('float32')
ref_pool['totalPrize_y']     = ref_pool['totalPrize_y'].astype('float32')

# Zu füllende Zeilen
null_rows = runners[runners['handicapRatingKg'].isna()][
    ['horseId', 'pos_perc_bin', 'age_join', 'class_join',
     'type_join', 'totalPrize_y', 'date']
].copy()
null_rows['_orig_index']  = null_rows.index
null_rows['totalPrize_y'] = null_rows['totalPrize_y'].astype('float32')

JOIN_KEYS  = ['pos_perc_bin', 'age_join', 'class_join', 'type_join']
CHUNK_SIZE = 10_000
results    = []

for start in tqdm(range(0, len(null_rows), CHUNK_SIZE), desc='Filling handicapRatingKg'):
    chunk = null_rows.iloc[start : start + CHUNK_SIZE]

    merged = chunk.merge(
        ref_pool[JOIN_KEYS + ['horseId', 'totalPrize_y',
                              'handicapRatingKg', 'first_rating_date', 'date']],
        on=JOIN_KEYS,
        suffixes=('_runner', '_ref')
    )

    # Kein Future-Leak
    merged = merged[merged['first_rating_date'] < merged['date_runner']]

    # Nicht dasselbe Pferd
    merged = merged[merged['horseId_runner'] != merged['horseId_ref']]

    # Prize-Toleranz
    prize_ok = merged['totalPrize_y_ref'].between(
        merged['totalPrize_y_runner'] * (1 - TOL),
        merged['totalPrize_y_runner'] * (1 + TOL)
    )
    merged = merged[prize_ok]

    if not merged.empty:
        agg_chunk = (
            merged
            .groupby('_orig_index')['handicapRatingKg']
            .agg(fill_value='mean', fill_n='count')
        )
        results.append(agg_chunk)

    del merged
    gc.collect()

# Ergebnisse zurückschreiben
if results:
    combined = pd.concat(results)
    agg = combined.groupby(level=0).agg(
        fill_value=('fill_value', 'mean'),
        fill_n=('fill_n', 'sum')
    )
    runners.loc[agg.index, 'handicapRatingKg'] = agg['fill_value'].astype('float32')

# Fehlende nach dem Fill flaggen
runners['handicapRatingKg_missing'] = runners['handicapRatingKg'].isna().astype('int8')

runners = runners.drop(
    columns=['pos_perc_bin', 'class_join', 'age_join', 'type_join'],
    errors='ignore'
)

del ref_pool, null_rows, results
gc.collect()


# ── ARR ───────────────────────────────────────────────────────────────────────

def calculate_arr(runners: pd.DataFrame) -> pd.DataFrame:
    runners = runners.copy()
    runners['ARR'] = np.nan

    groups = list(runners.groupby('raceId'))

    for race_id, race in tqdm(groups, desc='Calculating ARR'):
        going   = race['going_category'].iloc[0]
        d_group = race['distance_group'].iloc[0]

        if pd.isna(going) or pd.isna(d_group):
            continue
        if going not in kg_per_length or d_group not in kg_per_length[going]:
            continue

        kpl  = kg_per_length[going][d_group]
        mask = (race['pos_perc'].values > 0.66)

        if not mask.any():
            continue

        clb  = np.maximum(race['cumulative_lengths_back'].values, 0)
        wkg  = race['weightKg'].values
        hcap = race['handicapRatingKg'].values

        # Race überspringen wenn Referenzpferde (Top 33%) keine Rating haben
        if np.any(np.isnan(hcap[mask])):
            continue

        length_diff = clb[mask][:, None] - clb[None, :]

        with np.errstate(divide='ignore', invalid='ignore'):
            log_factor = np.where(
                length_diff > 0,
                np.maximum(np.log(length_diff) / np.log(4), 1.0),
                1.0
            )

        weight_diff = wkg[None, :] - wkg[mask][:, None]
        hcap_r      = hcap[mask][:, None]
        ref_matrix  = length_diff * kpl / log_factor + weight_diff * kpl + hcap_r

        ref_indices = np.where(mask)[0]
        all_indices = np.arange(len(race))
        diag_mask   = (ref_indices[:, None] == all_indices[None, :])
        ref_matrix  = np.where(diag_mask, hcap_r, ref_matrix)

        arr_vals = np.nanmean(ref_matrix, axis=0)
        arr_vals = np.where(np.isnan(arr_vals), np.nan, np.maximum(arr_vals, 0))

        runners.loc[race.index, 'ARR'] = np.round(arr_vals, 2)

    return runners

runners = calculate_arr(runners)

Calculating ARR: 100%|██████████| 28895/28895 [00:29<00:00, 983.55it/s] 


### 6.4  Derived Columns

In [43]:
# Draw fingerprint (for draw-bias features)
runners['draw_unique'] = (
    runners[['draw', 'name_meeting', 'distance', 'direction', 'rail', 'surface']]
    .apply(lambda r: ' - '.join(str(x) for x in r if pd.notna(x)), axis=1)
)

# Prize money by finishing position
PRIZE_SHARES = {1: 0.50, 2: 0.20, 3: 0.10, 4: 0.05}

def prize_money(row):
    share = PRIZE_SHARES.get(int(row['ranking']), 0.0) if pd.notna(row['ranking']) else 0.0
    return share * row.get('totalPrize_y', row.get('totalPrize', 0))

runners['prizemoney'] = runners.apply(prize_money, axis=1)

# Handicap rating adjusted for weight
runners['handicapRating_adj'] = runners['handicapRatingKg'] + 0.625 * (55 - runners['weightKg'])

# ── ARR-relative features ─────────────────────────────────────────────────────
_arr_col = 'handicapRating_adj'

# Gap to best-rated horse in the race (always ≤ 0; 0 = top-rated).
runners['arr_vs_field_max'] = (
    runners[_arr_col] - runners.groupby('raceId')[_arr_col].transform('max')
)

# ── Field-strength features ───────────────────────────────────────────────────
# Available at race time (handicapRating_adj is known before the race).
runners['field_avg_rating'] = runners.groupby('raceId')[_arr_col].transform('mean')
runners['field_std_rating'] = runners.groupby('raceId')[_arr_col].transform('std').fillna(0)
# Horse's rating relative to the field mean (signed; > 0 = above average)
runners['rating_vs_field_mean'] = runners[_arr_col] - runners['field_avg_rating']

# ── Market features ───────────────────────────────────────────────────────────
runners['market_prob'] = np.where(
    runners['liveOdd'].notna() & (runners['liveOdd'] > 0),
    1.0 / runners['liveOdd'],
    np.nan
)
runners['market_rank_pct'] = (
    runners.groupby('raceId')['market_prob']
    .rank(method='min', ascending=True, na_option='keep')
    .sub(1)
    .div(runners.groupby('raceId')['market_prob'].transform('count') - 1)
)

print(f'ARR proxy: {_arr_col!r}  |  field_avg/std/vs_mean + market features added.')

ARR proxy: 'handicapRating_adj'  |  field_avg/std/vs_mean + market features added.


### 6.5  Rolling Statistics (30D / 365D / 9999D)

In [44]:
stats_predictors = []

def add_rolling_stats(df, entity_col, windows, metrics=('prizemoney', 'pos_perc'), suffix_map=None):
    for window in windows:
        suffix = suffix_map.get(window, window) if suffix_map else window
        stat_base = (
            df.groupby(['date', entity_col])[list(metrics)]
            .mean().reset_index()
        )
        rolled = (
            stat_base.set_index('date')
            .groupby(entity_col)
            .rolling(window, closed='left')[list(metrics)]
            .mean().reset_index()
            .rename(columns={m: f'{entity_col}_{m}{suffix}' for m in metrics})
        )
        df = df.merge(rolled, on=['date', entity_col], how='left')
        for m in metrics:
            col = f'{entity_col}_{m}{suffix}'
            if col not in stats_predictors:
                stats_predictors.append(col)
    return df

print('horseSir (9999D)...')
runners = add_rolling_stats(runners, 'horseSir', ['9999D'])

print('draw_unique (9999D)...')
stat_du = runners.groupby(['date', 'draw_unique'])[['pos_perc']].mean().reset_index()
rolled_du = (
    stat_du.set_index('date')
    .groupby('draw_unique')
    .rolling('9999D', closed='left')[['pos_perc']]
    .mean().reset_index()
    .rename(columns={'pos_perc': 'draw_unique_posperc9999D'})
)
runners = runners.merge(rolled_du, on=['date', 'draw_unique'], how='left')
stats_predictors.append('draw_unique_posperc9999D')

for entity in ['trainerName', 'jockeyName', 'horseName', 'ownerName']:
    print(f'{entity} (30D, 365D)...')
    runners = add_rolling_stats(runners, entity, ['30D', '365D'])

# ── Win-rate: jockey (365D) and horse (365D + 730D) ──────────────────────────
# Rolling mean of win — mirrors josr365 / hosr730 from Benter reference
print('Win rates (jockey 365D, horse 365D+730D+90D)...')
for entity, windows in [('jockeyName', ['365D']), ('horseName', ['365D', '730D', '90D'])]:
    for window in windows:
        col_name = f'{entity}_winrate{window}'
        wr_base = runners.groupby(['date', entity])[['win']].mean().reset_index()
        wr_rolled = (
            wr_base.set_index('date')
            .groupby(entity)
            .rolling(window, closed='left')[['win']]
            .mean().reset_index()
            .rename(columns={'win': col_name})
        )
        runners = runners.merge(wr_rolled, on=['date', entity], how='left')
        stats_predictors.append(col_name)

print(f'\nRolling stats done — {len(stats_predictors)} predictors added.')


horseSir (9999D)...
draw_unique (9999D)...
trainerName (30D, 365D)...
jockeyName (30D, 365D)...
horseName (30D, 365D)...
ownerName (30D, 365D)...
Win rates (jockey 365D, horse 365D+730D+90D)...

Rolling stats done — 23 predictors added.


### 6.6  Percentile-rank Prize Money per Race

In [45]:
for entity in ['trainerName', 'horseName']:
    col_src = f'{entity}_prizemoney365D'
    col_out = f'{entity}_prizemoney365D_race_pct'
    race_mean = (
        runners.groupby('raceId')[col_src]
        .mean()
        .reset_index()
        .rename(columns={col_src: col_out})
    )
    runners = runners.merge(race_mean, on='raceId', how='left')
    stats_predictors.append(col_out)

print('Race-level percentile features done.')


Race-level percentile features done.


### 6.7  Context-Specific Preference Features (T-score)

In [46]:
stats_predictors_gtd = []

CONTEXT_COMBOS = {
    'trainerName': ['name_meeting', 'jockeyName', 'type'],
    'jockeyName':  ['name_meeting', 'trainerName', 'going_category'],
    'horseId':     ['name_meeting', 'trainerName', 'going_category', 'jockeyName', 'distance_group'],
    'horseSir':    ['going_category', 'distance_group'],
}
THRESHOLD_MAP = {'trainerName': 10, 'jockeyName': 10, 'horseId': 3, 'horseSir': 10}

# --- First pass: overall (entity) rolling stats ---
print('Calculating overall entity rolling stats...')
for entity in CONTEXT_COMBOS:
    window = '9999D' if entity == 'horseSir' else '10950D'

    overall = (
        runners.groupby(['date', entity])
        .agg(run=('ranking', 'count'), pos_perc_sum=('pos_perc', 'sum'))
        .reset_index()
    )
    overall_rolled = (
        overall.set_index('date')
        .groupby(entity)
        .rolling(window, closed='left')
        .agg({'pos_perc_sum': 'sum', 'run': 'sum'})
        .reset_index()
    )
    col_overall     = f'{entity}_pos_perc{window}'
    col_overall_run = f'{entity}_run{window}'
    overall_rolled[col_overall]     = overall_rolled['pos_perc_sum'] / overall_rolled['run']
    overall_rolled[col_overall_run] = overall_rolled['run']
    overall_rolled = overall_rolled[['date', entity, col_overall, col_overall_run]]

    # Drop before merge to avoid _x/_y suffix collision with columns from earlier cells
    runners = runners.drop(columns=[c for c in [col_overall, col_overall_run] if c in runners.columns])
    runners = runners.merge(overall_rolled, on=['date', entity], how='left')

# --- Second pass: context-specific stats and t-scores ---
print('\nCalculating context-specific rolling stats and t-scores...')
for entity, contexts in CONTEXT_COMBOS.items():
    window    = '9999D' if entity == 'horseSir' else '10950D'
    threshold = THRESHOLD_MAP[entity]
    col_overall     = f'{entity}_pos_perc{window}'
    col_overall_run = f'{entity}_run{window}'

    for ctx in contexts:
        print(f'  {entity} x {ctx}  ({window})')

        col_specific = f'{entity}{ctx}_pos_perc{window}'
        col_runs     = f'{entity}{ctx}_run{window}'

        spec = (
            runners.groupby(['date', entity, ctx])
            .agg(run=('ranking', 'count'), pos_perc_sum=('pos_perc', 'sum'))
            .reset_index()
        )
        spec_rolled = (
            spec.set_index('date')
            .groupby([entity, ctx])
            .rolling(window, closed='left')
            .agg({'pos_perc_sum': 'sum', 'run': 'sum'})
            .reset_index()
            .rename(columns={'run': col_runs})
        )
        spec_rolled[col_specific] = spec_rolled['pos_perc_sum'] / spec_rolled[col_runs]
        spec_rolled = spec_rolled[['date', entity, ctx, col_specific, col_runs]]

        # Drop before merge to avoid _x/_y suffix collision
        runners = runners.drop(columns=[c for c in [col_specific, col_runs] if c in runners.columns])
        runners = runners.merge(spec_rolled, on=['date', entity, ctx], how='left')

        col_tscore = f'{entity}{ctx}_tscore{window}'
        runners[col_tscore] = np.where(
            runners[col_runs] >= threshold,
            (runners[col_specific] - runners[col_overall]) * np.sqrt(runners[col_runs]),
            np.nan
        )
        stats_predictors_gtd.append(col_tscore)

print(f'\nContext preference features done — {len(stats_predictors_gtd)} features.')

Calculating overall entity rolling stats...

Calculating context-specific rolling stats and t-scores...
  trainerName x name_meeting  (10950D)
  trainerName x jockeyName  (10950D)
  trainerName x type  (10950D)
  jockeyName x name_meeting  (10950D)
  jockeyName x trainerName  (10950D)
  jockeyName x going_category  (10950D)
  horseId x name_meeting  (10950D)
  horseId x trainerName  (10950D)
  horseId x going_category  (10950D)
  horseId x jockeyName  (10950D)
  horseId x distance_group  (10950D)
  horseSir x going_category  (9999D)
  horseSir x distance_group  (9999D)

Context preference features done — 13 features.


In [47]:
# Load cached state
df_with_ratings = pd.read_parquet(f"{BASE_PATH}/df_with_ratings.parquet")

with open(f"{BASE_PATH}/ratings_state.json") as f:
    saved_ratings = {int(k): v for k, v in json.load(f).items()}

with open(f"{BASE_PATH}/ratings_last_date.txt") as f:
    last_processed_date = f.read().strip()

print(f"📂 Loaded {len(df_with_ratings)} cached rows (up to {last_processed_date})")

# Find new races not yet in the cache
new_runners = runners[
    (runners['date'] >= '2021-01-01') &
    (runners['date'] > last_processed_date) &
    (~runners['raceId'].isin(df_with_ratings['raceId']))
]

if new_runners.empty:
    print("✅ No new races — using cached ratings.")
else:
    print(f"⚙️  Processing {new_runners['raceId'].nunique()} new races ({len(new_runners)} rows)...")

    new_rated = update_ratings_with_history(
        new_runners, k_factor=0.5, initial_ratings=saved_ratings
    )
    new_rated['rating_after_race'] = new_rated['rating_after_race'].round(1)
    new_rated['draw_unique'] = new_rated.apply(
        make_draw_unique, axis=1, meeting_col='name_meeting'
    )

    # Append and save
    df_with_ratings = pd.concat([df_with_ratings, new_rated], ignore_index=True)
    df_with_ratings.to_parquet(f"{BASE}/df_with_ratings.parquet", index=False)

    # Update ratings state
    updated_ratings = (
        new_rated.sort_values('date')
        .groupby('horseId')['rating_after_race']
        .last()
        .to_dict()
    )
    saved_ratings.update({int(k): v for k, v in updated_ratings.items()})
    with open(f"{BASE_PATH}/ratings_state.json", "w") as f:
        json.dump({str(k): v for k, v in saved_ratings.items()}, f)

    with open(f"{BASE_PATH}/ratings_last_date.txt", "w") as f:
        f.write(str(df_with_ratings['date'].max()))

    print(f"✅ Appended {len(new_rated)} new rows. Total: {len(df_with_ratings)}")

runners = runners.merge(df_with_ratings[['raceId', 'horseId', 'rating_after_race']], on=['raceId', 'horseId'])

📂 Loaded 261606 cached rows (up to 2026-05-05 00:00:00)
✅ No new races — using cached ratings.


### 6.8  Last-Time-Out (LTO) Features

In [48]:
runners = runners.sort_values('date', ascending=True).reset_index(drop=True)
runners['horse_run'] = runners.groupby('horseId').cumcount()

LTO_BASE_COLS = [
    'cumulative_lengths_back', 'pos_perc', 'prizemoney',
    'liveOdd', 'totalPrize_y', 'type', 'date', 'blinkers',
] + [c for c in ['ARR', 'handicapRatingKg', 'rating_after_race'] if c in runners.columns]

LTO_LAGS     = [1, 2, 3]
LTO_CONTEXTS = ['going_category', 'distance_group', 'name_meeting']

LTO_EXCLUDE = {'type', 'date', 'blinkers'}   # used only for derived features

lto_predictors = []

for col in LTO_BASE_COLS:
    for lag in LTO_LAGS:
        tmp = runners[['horseId', 'horse_run', col]].copy()
        tmp['horse_run'] = tmp['horse_run'] + lag
        runners = runners.merge(
            tmp, on=['horseId', 'horse_run'], how='left', suffixes=['', f'_{lag}lto']
        )
        if col not in LTO_EXCLUDE:
            lto_predictors.append(f'{col}_{lag}lto')

for cat in LTO_CONTEXTS:
    for col in LTO_BASE_COLS:
        tmp = runners[['horseId', 'horse_run', cat, col]].copy()
        tmp['horse_run'] = tmp['horse_run'] + 1
        runners = runners.merge(
            tmp, on=['horseId', 'horse_run', cat], how='left',
            suffixes=['', f'_1lto_{cat}']
        )
        if col not in LTO_EXCLUDE:
            lto_predictors.append(f'{col}_1lto_{cat}')

for cat_a, cat_b in combinations(LTO_CONTEXTS, 2):
    combo = f'{cat_a}_{cat_b}'
    for col in ['pos_perc', 'cumulative_lengths_back']:
        tmp = runners[['horseId', 'horse_run', cat_a, cat_b, col]].copy()
        tmp['horse_run'] = tmp['horse_run'] + 1
        runners = runners.merge(
            tmp, on=['horseId', 'horse_run', cat_a, cat_b], how='left',
            suffixes=['', f'_1lto_{combo}']
        )
        lto_predictors.append(f'{col}_1lto_{combo}')

runners['datesince'] = (runners['date'] - runners['date_1lto']).dt.days

runners['pos_trend_3']     = runners['pos_perc_1lto']                - runners['pos_perc_3lto']
runners['lengths_trend_3'] = runners['cumulative_lengths_back_1lto'] - runners['cumulative_lengths_back_3lto']
runners['pos_trend_1v2']   = runners['pos_perc_1lto']                - runners['pos_perc_2lto']

_arr_lto1 = 'ARR_1lto' if 'ARR_1lto' in runners.columns else 'handicapRatingKg_1lto'
_arr_lto3 = 'ARR_3lto' if 'ARR_3lto' in runners.columns else 'handicapRatingKg_3lto'
if _arr_lto1 in runners.columns and _arr_lto3 in runners.columns:
    runners['arr_trend_3'] = runners[_arr_lto1] - runners[_arr_lto3]
    lto_predictors.append('arr_trend_3')

lto_predictors += ['pos_trend_3', 'lengths_trend_3', 'pos_trend_1v2']

# ── Mean of last 3 SPs (analog of homean4sprat in reference) ─────────────────
runners['liveOdd_mean3'] = runners[['liveOdd_1lto', 'liveOdd_2lto', 'liveOdd_3lto']].mean(axis=1)

# ── First-time blinkers flag ──────────────────────────────────────────────────
def _equip_on(col):
    return col.map(lambda x: str(x).lower() not in ('0', 'false', 'none', 'nan', 'non', 'no', ''))

runners['blinkers_1sttime'] = (
    _equip_on(runners['blinkers']) &
    (~_equip_on(runners['blinkers_1lto']) | runners['blinkers_1lto'].isna())
).astype(int)

# ── Last-run delta features ───────────────────────────────────────────────────
# Change in handicap rating since last run (positive = risen in weights).
runners['handicapRating_delta'] = runners['handicapRatingKg'] - runners['handicapRatingKg_1lto']
# Weight change since last run.
runners['weightKg_delta'] = runners['weightKg'] - (
    runners['weightKg_1lto'] if 'weightKg_1lto' in runners.columns else np.nan
)
# Odds drift: how much shorter/longer vs last run (negative = shorter = market warming).
runners['liveOdd_delta'] = runners['liveOdd'] - runners['liveOdd_1lto']

lto_predictors += ['handicapRating_delta', 'weightKg_delta', 'liveOdd_delta']

runners['ARR_3lto'] = runners['ARR_3lto'] + 55 -runners['weightKg']
runners['ARR_2lto'] = runners['ARR_2lto'] + 55 -runners['weightKg']
runners['ARR_1lto'] = runners['ARR_1lto'] + 55 -runners['weightKg']

# ── Variance / consistency features ──────────────────────────────────────────
# Standard deviation of pos_perc across last 3 runs (NaN if < 2 valid runs).
runners['pos_perc_std3'] = runners[
    ['pos_perc_1lto', 'pos_perc_2lto', 'pos_perc_3lto']
].std(axis=1)
lto_predictors.append('pos_perc_std3')

# ── First-time condition flags ────────────────────────────────────────────────
# NaN in the LTO-context column means no prior run in that condition -> first time.
runners['first_time_going']     = runners['pos_perc_1lto_going_category'].isna().astype(int)
runners['first_time_distance']  = runners['pos_perc_1lto_distance_group'].isna().astype(int)
runners['first_time_course']    = runners['pos_perc_1lto_name_meeting'].isna().astype(int)
lto_predictors += ['first_time_going', 'first_time_distance', 'first_time_course']

# ── Exponential-decay form score ──────────────────────────────────────────────
# Recency-weighted pos_perc: recent strong form counts more than stale form.
# Half-life ~ 42 days (exp(-60/60) = 0.37 at 60d, ~0.14 at 120d).
runners['form_decay60'] = (
    np.exp(-runners['datesince'].clip(lower=0) / 60.0) * runners['pos_perc_1lto']
)
lto_predictors.append('form_decay60')

print(f'LTO features done — {len(lto_predictors)} predictors.')
print(f'liveOdd_mean3 non-null: {runners["liveOdd_mean3"].notna().sum()}')
print(f'blinkers_1sttime count: {runners["blinkers_1sttime"].sum()}')
print(f'New delta/consistency/flags added.')

LTO features done — 66 predictors.
liveOdd_mean3 non-null: 215180
blinkers_1sttime count: 16183
New delta/consistency/flags added.


### 6.9  Web Tips Rank

In [49]:
def process_webtips_fast(webTips_df: pd.DataFrame, runners_df: pd.DataFrame) -> pd.DataFrame:
    """Vectorised webTips ranking — assigns webTipRank and webTipRank_perc."""
    runners_df = runners_df.copy()
    runners_df['webTipRank']      = np.nan
    runners_df['webTipRank_perc'] = np.nan

    # Build a lookup: raceId → {horseName: rank}
    # Process all tips at once, grouped by raceId
    tip_lookup = {}  # raceId -> {horseName: rank}

    for _, tip_row in webTips_df.iterrows():
        if pd.isna(tip_row.get('text')):
            continue
        rid = tip_row['raceId']
        if rid in tip_lookup:
            continue  # one tip per race, skip duplicates
        tip_lookup[rid] = tip_row['text']

    # Build a flat dataframe of (raceId, horseName, rank, rank_perc)
    records = []
    race_groups = runners_df.groupby('raceId')['horseName'].apply(list)

    for rid, text in tqdm(tip_lookup.items(), desc='webTips'):
        if rid not in race_groups.index:
            continue
        horse_names = race_groups[rid]
        lower       = text.lower()

        positions = []
        for horse in horse_names:
            match = re.search(r'\b' + re.escape(horse.lower()) + r'\b', lower)
            if match:
                positions.append((match.start(), horse))

        positions.sort()
        n = len(positions)
        for rank, (_, horse) in enumerate(positions, start=1):
            records.append({
                'raceId':         rid,
                'horseName':      horse,
                'webTipRank':     rank,
                'webTipRank_perc': (n - rank) / (n - 1) if n > 1 else 0.5,
            })

    if not records:
        return runners_df

    ranks_df = pd.DataFrame(records)

    # Single merge instead of per-row mask assignments
    runners_df = runners_df.drop(columns=['webTipRank', 'webTipRank_perc'])
    runners_df = runners_df.merge(ranks_df, on=['raceId', 'horseName'], how='left')

    return runners_df

runners = process_webtips_fast(webTips, runners)
print('webTipRank_perc computed.')

webTips: 100%|██████████| 24313/24313 [00:12<00:00, 1901.07it/s]


webTipRank_perc computed.


### 6.10  Odds & Place SP

In [50]:
runners['odds']         = 1 / runners['liveOdd']
runners['odds_sum']     = runners.groupby(['raceId', 'name_meeting'])['odds'].transform('sum')
runners['place_sp']     = (runners['liveOdd'] - 1) / 4 + 1
runners['odds_place']   = 1 / runners['place_sp']

# Adjust place odds to target n_places × PLACE_MARGIN
n_places        = runners.groupby('raceId')['place'].transform('sum')
sum_place_odds  = runners.groupby('raceId')['odds_place'].transform('sum')
runners['odds_place'] *= (n_places * PLACE_MARGIN) / sum_place_odds

# Additional race-level metadata
runners = runners.merge(
    races[['id_race', 'minAge']], left_on='raceId', right_on='id_race', how='left'
)
print('Odds features done.')

Odds features done.


## 7  Modelling

### 7.1  Build Feature Lists & Filter Dataset

In [51]:
# ── Feature sets ─────────────────────────────────────────────────────────────
# Single-model approach (reference paper style): train one LightGBM classifier
# on all fundamental features PLUS liveOdd.  The model learns the residual
# signal on top of market odds; EV>0 horses are where we disagree with the market.

TYPE_LTO_COLS = [c for c in lto_predictors if 'type_' not in c]

predictors_cat  = [
    'going_category', 'distance_group', 'type', 'sex', 'blinkers', 'tongueTie',
    'type_1lto', 'type_2lto', 'type_3lto',
]
predictors_misc = ['runners', 'age', 'datesince', 'webTipRank_perc']

predictors_field = [
    'field_avg_rating', 'field_std_rating', 'rating_vs_field_mean',
]

FUND_FEATURES = (
    stats_predictors
    + TYPE_LTO_COLS
    + ['weightKg', 'handicapRating_adj', 'arr_vs_field_max']
    + predictors_field
    + stats_predictors_gtd
    + predictors_cat
    + predictors_misc
)

# Include liveOdd: mirrors the reference code which uses odds as a feature.
# The single-model approach learns the residual signal on top of market odds —
# when p_model > p_market the horse has positive EV.
ALL_FEATURES = FUND_FEATURES + [
    'liveOdd',
    'market_rank_pct',   # within-race rank: 0=favourite, 1=outsider
]

# ── Dataset split by mode ─────────────────────────────────────────────────────
has_odds = runners['liveOdd'].notna()
in_band  = runners['odds_sum'].between(ODDS_SUM_MIN, ODDS_SUM_MAX)

if MODE == 'train':
    df = runners[has_odds & in_band & runners['date'].between(TRAIN_START, TRAIN_END)].copy()
elif MODE == 'test':
    df = runners[has_odds & in_band & (runners['date'] >= TRAIN_END)].copy()
else:
    df = runners[runners['date'] >= TODAY].copy()

print(f'Dataset size: {df.shape}  |  Fund features: {len(FUND_FEATURES)}')
print(f'Positive class (place=1): {df["place"].sum() if "place" in df.columns else "N/A"}')

Dataset size: (133956, 260)  |  Fund features: 123
Positive class (place=1): 35844


In [52]:
# -- 7.1b  Within-race relative features ------------------------------------
# LightGBM sees absolute values: a jockey with 45% win-rate looks the same
# whether they're the best or worst jockey in THIS race.
# Within-race rank features give the model direct access to relative comparisons.
# rank_pct: 0.0 = best in race, 1.0 = worst in race.
# NaN (no history) stays NaN -- model handles natively.

# All rolling stats (stats_predictors) are higher-is-better.
# All context t-scores (stats_predictors_gtd) are higher-is-better.
# LTO and other numeric features have their own directions.
_auto_higher = (
    [(col, True) for col in stats_predictors     if col in df.columns]
  + [(col, True) for col in stats_predictors_gtd if col in df.columns]
)

_manual = [
    ('handicapRating_adj',           True),   # higher rating = better
    ('rating_vs_field_mean',         True),   # above-average rated
    ('form_decay60',                 True),   # recent form
    ('pos_perc_1lto',                True),
    ('pos_perc_2lto',                True),
    ('pos_perc_3lto',                True),
    ('cumulative_lengths_back_1lto', False),  # less lengths back = better
    ('cumulative_lengths_back_2lto', False),
    ('liveOdd_delta',                False),  # negative = shortening = warming
    ('liveOdd',                      False),  # lower = favourite
    ('webTipRank_perc',              True),
    ('age',                          False),  # younger generally better in FR flat
    ('weightKg',                     False),  # lower weight = advantage
    ('datesince',                    False),  # lower = fresher
]

# Deduplicate (manual overrides auto if same col appears in both)
_seen = set()
_RANK_COLS = []
for col, direction in _manual + _auto_higher:
    if col in df.columns and col not in _seen:
        _RANK_COLS.append((col, direction))
        _seen.add(col)

_rank_predictors = []
for _col, _higher_better in _RANK_COLS:
    _out = f'{_col}_racerank'
    _asc = not _higher_better
    _n   = df.groupby('raceId')[_col].transform('count').clip(lower=2)
    df[_out] = (
        df.groupby('raceId')[_col]
        .rank(method='average', ascending=_asc, na_option='keep') - 1
    ) / (_n - 1)
    _rank_predictors.append(_out)

ALL_FEATURES = ALL_FEATURES + _rank_predictors
print(f'Within-race rank features: {len(_rank_predictors)}')
print(f'Total features: {len(ALL_FEATURES)}')


Within-race rank features: 52
Total features: 177


### 7.2  Encode Categoricals

In [53]:
le = LabelEncoder()
# Encode cats that are still used for diagnostic labels
for col in ['going_category', 'distance_group', 'type', 'sex', 'blinkers', 'tongueTie',
            'type_1lto', 'type_2lto', 'type_3lto']:
    if col in df.columns:
        df[col] = le.fit_transform(df[col].astype(str))

df['going_label']    = runners.loc[df.index, 'going_category']
df['racetype_label'] = runners.loc[df.index, 'type']

df.replace([np.inf, -np.inf], np.nan, inplace=True)


/tmp/ipykernel_15893/2679582679.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace([np.inf, -np.inf], np.nan, inplace=True)


### 7.3  Group-wise Scaling (within each race)

In [54]:
# Group-wise scaling removed: LinearCLogit learns directly on raw feature values.
# The conditional logistic regression loss is scale-invariant in the sense that
# the optimiser will find appropriate weights for each feature's natural scale.
scaled_cols = []
print('Group-wise scaling skipped — using raw features.')


Group-wise scaling skipped — using raw features.


### 7.4  Feature Matrix & Target

In [55]:
X = df[ALL_FEATURES + ['raceId', 'odds_sum', 'date']]
y = df[['place', 'win']].copy()

print(f'X shape: {X.shape}  |  y shape: {y.shape}')
print(f'Positive class (place=1): {y["place"].sum()}  ({100*y["place"].mean():.1f}%)')
print(f'Positive class (win=1):   {y["win"].sum()}  ({100*y["win"].mean():.1f}%)')


X shape: (133956, 180)  |  y shape: (133956, 2)
Positive class (place=1): 35844  (26.8%)
Positive class (win=1):   12353  (9.2%)


### 7.5  Train/Test Split & Outlier Removal

In [56]:
if MODE == 'train':
    _dates     = X['date'].sort_values()
    _cutoff    = _dates.quantile(0.8)
    train_mask = df['date'] < _cutoff
    test_mask  = df['date'] >= _cutoff

    X_train, X_test = X[train_mask], X[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]

    print(f'Cutoff: {_cutoff.date()}')
    print(f'Train: {X_train.shape} ({train_mask.sum()} rows)')
    print(f'Test:  {X_test.shape}  ({test_mask.sum()} rows)')


Cutoff: 2025-02-14
Train: (107080, 180) (107080 rows)
Test:  (26876, 180)  (26876 rows)


### 7.6  Global Scaling (before LightGBM)

In [57]:
# -- 7.6  Prepare feature matrix for LightGBM --------------------------------
# LightGBM is scale-invariant (tree splits on thresholds, not magnitudes),
# so StandardScaler is unnecessary and removed.
#
# NaN handling:
#   - datesince NaN -> first-time runner, fill 999 (very long since last run)
#   - webTipRank_perc NaN -> not tipped, fill 0.0 (unfancied)
#   - ALL other NaN -> let LightGBM decide natively (learns optimal direction
#     per split; filling with 0 incorrectly labels "no history" as "worst value")

_FILL_MAP = {
    'datesince':      999,   # no previous run
    'webTipRank_perc': 0.0,  # not mentioned by tipsters
}

def _prepare_X(df_subset, cols, fill_map=_FILL_MAP):
    X_out = df_subset[cols].copy()
    for col, val in fill_map.items():
        if col in X_out.columns:
            X_out[col] = X_out[col].fillna(val)
    # Cast to float32; NaN is valid in float32 and LightGBM handles it natively
    return X_out.astype(np.float32)

if MODE == 'train':
    X_train_final = _prepare_X(X_train, ALL_FEATURES)
    X_test_final  = _prepare_X(X_test,  ALL_FEATURES)
    X_full_final  = _prepare_X(X,       ALL_FEATURES)
else:
    X_full_final  = _prepare_X(X, ALL_FEATURES)

_nan_pct = X_train_final.isna().mean() if MODE == 'train' else X_full_final.isna().mean()
_top_nan = _nan_pct.sort_values(ascending=False).head(5)
print(f'Feature matrix ready: {X_train_final.shape if MODE == "train" else X_full_final.shape}')
print(f'Top-5 NaN% (passed to LightGBM natively):')
for col, pct in _top_nan.items():
    if pct > 0:
        print(f'  {col}: {100*pct:.1f}%')


Feature matrix ready: (107080, 177)
Top-5 NaN% (passed to LightGBM natively):
  weightKg_delta: 100.0%
  cumulative_lengths_back_1lto_distance_group_name_meeting: 87.9%
  pos_perc_1lto_distance_group_name_meeting: 87.9%
  pos_perc_1lto_going_category_name_meeting: 84.7%
  cumulative_lengths_back_1lto_going_category_name_meeting: 84.7%


### 7.7  Train LightGBM

In [58]:
# -- 7.7  Single LightGBM Classifier (reference paper architecture) ----------
#
# One model trained on ALL features including liveOdd (market odds).
# Mirrors the reference: clogit(fundamental_features + odds).
# The model learns residual signal on top of what odds already imply.
# When p_model > p_market -> positive EV -> bet.
#
# No two-step blend needed: market information is already in the features.

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

def _race_normalize(scores, race_ids):
    """Normalize raw scores to sum to 1 within each race (like reference softmax)."""
    probs = np.zeros(len(scores), dtype=np.float64)
    for rid in np.unique(race_ids):
        m = race_ids == rid
        s = scores[m].sum()
        if s > 0:
            probs[m] = scores[m] / s
    return probs

def _market_probs_np(liveodds_np, race_ids_np):
    _raw = np.where(np.isnan(liveodds_np) | (liveodds_np <= 0), 1e-6, 1.0 / liveodds_np)
    probs = np.zeros_like(_raw)
    for rid in np.unique(race_ids_np):
        m = race_ids_np == rid
        s = _raw[m].sum()
        if s > 0: probs[m] = _raw[m] / s
    return probs

# Placeholders for backward-compat references in later cells
model       = None
model_win   = None
fund_net    = None
fund_ranker = None
fund_clf    = None
blend_net   = None

if MODE == 'train':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    n_feat = X_train_final.shape[1]
    print(f'Device: {device}  |  Features: {n_feat}')

    _train_rids = df.loc[X_train.index, 'raceId'].values
    _test_rids  = df.loc[X_test.index,  'raceId'].values
    _full_rids  = df['raceId'].values

    print('\n-- LightGBM Classifier  [WIN target, liveOdd included] -------')

    fund_clf = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.02,
        num_leaves=63,
        min_child_samples=80,
        feature_fraction=0.7,
        bagging_fraction=0.8,
        bagging_freq=5,
        reg_alpha=0.05,
        reg_lambda=0.1,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    fund_clf.fit(
        X_train_final, y_train['win'],
        feature_name=ALL_FEATURES,
        eval_set=[(X_test_final, y_test['win'])],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
    )

    # Feature importance
    _imp = fund_clf.booster_.feature_importance(importance_type='gain')
    _feat_imp = sorted(zip(ALL_FEATURES, _imp), key=lambda x: x[1], reverse=True)
    print('\nTop-20 features by gain:')
    for _fn, _fw in _feat_imp[:20]:
        print(f'  {_fw:10.1f}  {_fn}')
    _zero = sum(1 for _, w in _feat_imp if w == 0)
    print(f'\nUsed: {n_feat - _zero}/{n_feat}  |  Best iter: {fund_clf.best_iteration_}')

    # Per-race normalized probabilities (mirrors reference softmax)
    _r_train = fund_clf.predict_proba(X_train_final)[:, 1]
    _r_test  = fund_clf.predict_proba(X_test_final)[:, 1]

    p_model_train = _race_normalize(_r_train, _train_rids)
    p_model_test  = _race_normalize(_r_test,  _test_rids)

    auc_win = roc_auc_score(y_test['win'],   _r_test)
    auc_plc = roc_auc_score(y_test['place'], _r_test)
    _best_auc = fund_clf.best_score_['valid_0']['auc']
    print(f'\nAUC  win={auc_win:.4f}  place={auc_plc:.4f}  (val win AUC: {_best_auc:.4f})')

    # Market-only AUC for comparison
    _test_lo = df.loc[X_test.index, 'liveOdd'].values
    p_mkt_test = _market_probs_np(_test_lo, _test_rids)
    auc_mkt_win = roc_auc_score(y_test['win'],   p_mkt_test)
    auc_mkt_plc = roc_auc_score(y_test['place'], p_mkt_test)
    print(f'Market-only AUC  win={auc_mkt_win:.4f}  place={auc_mkt_plc:.4f}')

    # -- OOS sim: EV strategy (reference paper approach) ---------------------
    _sim = df.loc[X_test.index, ['raceId', 'win', 'liveOdd', 'place', 'place_sp']].copy()
    _sim['p_model'] = p_model_test

    # Win EV  (reference formula: p*(odds-1) - (1-p))
    _sim['ev_win']   = _sim['p_model'] * (_sim['liveOdd']  - 1) - (1 - _sim['p_model'])
    # Place EV
    _sim['ev_place'] = _sim['p_model'] * (_sim['place_sp'] - 1) - (1 - _sim['p_model'])

    def _ev_sim(sim_df, ev_col, sp_col, win_col, label):
        _pos = sim_df[sim_df[ev_col] > 0]
        _bets = _pos.loc[_pos.groupby('raceId')[ev_col].idxmax()]
        n = len(_bets)
        if n == 0:
            print(f'  {label}: 0 bets')
            return
        roi = 100 * (_bets[_bets[win_col] == 1][sp_col].sum() / n - 1)
        sr  = 100 * (_bets[win_col] == 1).mean()
        print(f'  {label}: Bets={n}  SR={sr:.1f}%  ROI={roi:+.2f}%')
        for band, lo, hi in [('<3', 0, 3), ('3-6', 3, 6), ('>6', 6, 999)]:
            g = _bets[(_bets[sp_col] >= lo) & (_bets[sp_col] < hi)]
            if len(g) < 5: continue
            r = 100 * (g[g[win_col] == 1][sp_col].sum() / len(g) - 1)
            s = 100 * (g[win_col] == 1).mean()
            print(f'    {band}: n={len(g)}  SR={s:.1f}%  ROI={r:+.2f}%')

    # Also top-pick strategy for comparison
    _sim['rank'] = _sim.groupby('raceId')['p_model'].rank(ascending=False, method='first')
    _top = _sim[_sim['rank'] == 1]
    _roi_w = 100 * (_top[_top['win']   == 1]['liveOdd'].sum()  / len(_top) - 1)
    _roi_p = 100 * (_top[_top['place'] == 1]['place_sp'].sum() / len(_top) - 1)
    print(f'\nOOS top-pick (n={len(_top)}):  ROI win={_roi_w:+.2f}%  ROI place={_roi_p:+.2f}%')

    print('\nOOS EV strategy:')
    _ev_sim(_sim, 'ev_win',   'liveOdd',  'win',   'Win  EV bets')
    _ev_sim(_sim, 'ev_place', 'place_sp', 'place', 'Place EV bets')


Device: cpu  |  Features: 177

-- LightGBM Classifier  [WIN target, liveOdd included] -------
[100]	valid_0's auc: 0.77694	valid_0's binary_logloss: 0.260908
[200]	valid_0's auc: 0.778253	valid_0's binary_logloss: 0.258656

Top-20 features by gain:
    143192.0  liveOdd
     14273.5  liveOdd_racerank
     12486.3  market_rank_pct
      9791.6  rating_vs_field_mean
      6547.3  webTipRank_perc
      4820.3  handicapRating_adj_racerank
      4152.5  liveOdd_delta
      3030.4  weightKg_racerank
      2917.4  pos_trend_3
      2812.0  handicapRating_adj
      2719.9  arr_vs_field_max
      2403.0  rating_after_race_3lto
      2335.0  cumulative_lengths_back_3lto
      2288.1  liveOdd_1lto
      2263.4  liveOdd_3lto
      2198.8  lengths_trend_3
      2155.5  trainerNamename_meeting_tscore10950D
      2127.0  runners
      2096.5  draw_unique_posperc9999D
      2058.8  horseSirgoing_category_tscore9999D

Used: 171/177  |  Best iter: 205

AUC  win=0.7785  place=0.7699  (val win AUC: 0.7785

In [59]:
# ── 7.7b  Optuna Hyperparameter Search  ──────────────────────────────────────
# Set OPTUNA_ENABLED=True in config to run.  Takes ~30 min for 100 trials.
# Optimises the binary classifier — objective is AUC on the OOS test set.

if MODE == 'train' and OPTUNA_ENABLED:
    if not _OPTUNA_AVAILABLE:
        import subprocess
        subprocess.run(['pip', 'install', 'optuna', '-q'])
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)

    _n_pos_opt = y_train['place'].sum()
    _sp_opt    = (len(y_train) - _n_pos_opt) / _n_pos_opt
    _lo_opt    = df.loc[X_train.index, 'liveOdd'].fillna(1.0).values
    _sw_opt    = np.clip(np.where(y_train['place'].values == 1, _lo_opt, 1.0), 1.0, 20.0)

    def _optuna_objective(trial):
        params = dict(
            n_estimators      = trial.suggest_int('n_estimators', 300, 3000),
            learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            num_leaves        = trial.suggest_int('num_leaves', 16, 128),
            min_child_samples = trial.suggest_int('min_child_samples', 20, 200),
            subsample         = trial.suggest_float('subsample', 0.5, 1.0),
            subsample_freq    = 1,
            colsample_bytree  = trial.suggest_float('colsample_bytree', 0.4, 1.0),
            reg_alpha         = trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
            reg_lambda        = trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
            scale_pos_weight  = _sp_opt,
            random_state=42, n_jobs=-1,
        )
        m = lgb.LGBMClassifier(**params)
        m.fit(
            X_train_final, y_train['place'],
            sample_weight=_sw_opt,
            eval_set=[(X_test_final, y_test['place'])],
            callbacks=[lgb.early_stopping(30, verbose=False)],
        )
        return roc_auc_score(y_test['place'], m.predict_proba(X_test_final)[:, 1])

    study = optuna.create_study(direction='maximize')
    study.optimize(_optuna_objective, n_trials=100, timeout=1800)

    print(f'Best AUC: {study.best_value:.4f}')
    print('Best params:', study.best_params)

    model = lgb.LGBMClassifier(**study.best_params, random_state=42, n_jobs=-1)
    model.fit(
        X_train_final, y_train['place'],
        sample_weight=_sw_opt,
        eval_set=[(X_test_final, y_test['place'])],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)],
    )
    preds = model.predict_proba(X_test_final)[:, 1]
    print(f'Retrained — AUC: {roc_auc_score(y_test["place"], preds):.4f}')

    # Recalibrate threshold with the tuned model
    prec, rec, thresholds = precision_recall_curve(y_test['place'], preds)
    _f1s = 2 * prec * rec / (prec + rec + 1e-8)
    best_thresh = float(thresholds[np.argmax(_f1s[:-1])])
    print(f'Best threshold (F1): {best_thresh:.4f}')

[W 2026-05-07 08:30:30,515] Trial 2 failed with parameters: {'n_estimators': 362, 'learning_rate': 0.05821672362294676, 'num_leaves': 64, 'min_child_samples': 172, 'subsample': 0.8246377331380487, 'colsample_bytree': 0.5966683355649456, 'reg_alpha': 0.027067137493335686, 'reg_lambda': 1.8078654628766166} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_15893/1293951882.py", line 32, in _optuna_objective
    m.fit(
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py", line 1560, in fit
    super().fit(
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py", line 1049, in fit
    self._Booster = train(
                    ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/engine.py", line 297, in train
    booster = Boo

KeyboardInterrupt: 

In [ ]:
# ── 7.7c  Win Model ───────────────────────────────────────────────────────────
# Disabled: the softmax rating network in 7.7 already trains directly on win
# outcomes using win-log-loss, so a separate win model is not needed.
model_win = None

In [ ]:
import joblib

# joblib.dump(model,        f'{BASE_PATH}/pt_model.pkl')
# joblib.dump(global_scaler,f'{BASE_PATH}/pt_scaler.pkl')
# joblib.dump(ALL_FEATURES, f'{BASE_PATH}/pt_features.pkl')
# joblib.dump(best_thresh,  f'{BASE_PATH}/pt_best_thresh.pkl')
# if model_win is not None:
#     joblib.dump(model_win, f'{BASE_PATH}/pt_model_win.pkl')

# print("Model, scaler, features, threshold and win model saved.")


### 7.8  Predict

In [ ]:
# -- 7.8  Predict ---------------------------------------------------------
_full_rids = df['raceId'].values

_r_full = fund_clf.predict_proba(X_full_final)[:, 1]
label_preds_prob = _race_normalize(_r_full, _full_rids)

_max_dev = abs(
    pd.Series(label_preds_prob, index=df.index).groupby(df['raceId']).sum() - 1
).max()
print(f'Prob range: {label_preds_prob.min():.4f}-{label_preds_prob.max():.4f}  '
      f'mean={label_preds_prob.mean():.4f}  max race-sum deviation={_max_dev:.6f}')

label_preds = np.zeros(len(label_preds_prob), dtype=int)


## 8  Evaluation

In [ ]:
def calculate_win_profit(position: int, win_sp: float) -> float:
    return win_sp - 1 if position == 1 else -1

def build_evaluation_df(df, y_true, prob_scores, mode='train') -> pd.DataFrame:
    today = pd.Timestamp(datetime.today().date())

    eval_df = pd.DataFrame(y_true).copy()
    eval_df = eval_df.assign(
        place_sp        = df['place_sp'],
        win_sp          = df['liveOdd'],
        win_odds        = 1 / df['liveOdd'],
        horse           = df['horseName'],
        course          = df['name_meeting'],
        distype         = df['distance_group'],
        webTipRank_perc = df['webTipRank_perc'],
        going_category  = df.get('going_label',  df['going_category']),
        win             = df['win'],
        age             = df['age'],
        place           = df['place'],
        position        = df['ranking'],
        racetype        = df.get('racetype_label', df['type']),
        date            = df['date'],
        raceId          = df['raceId'],
        market_rank_pct = df['market_rank_pct'],
        y_pred_raw      = prob_scores,
    )

    # ── Model probability adjusted for overround ──────────────────────────────
    race_sum            = eval_df.groupby('raceId')['y_pred_raw'].transform('sum')
    eval_df['adj_prob'] = eval_df['y_pred_raw'] * (WIN_OVERROUND / race_sum)
    eval_df['SP']       = np.round(1 / eval_df['adj_prob'], 1)

    # ── Expected Value: EV = p_model × (odds−1) − (1−p_model)  [ref: Benter] ─
    # Positive EV → model thinks this horse is underpriced by the market.
    eval_df['ev_raw'] = eval_df['adj_prob'] * (eval_df['win_sp'] - 1) - (1 - eval_df['adj_prob'])

    eval_df['runners']  = eval_df.groupby('raceId')['horse'].transform('count')
    eval_df['rank']     = eval_df.groupby('raceId')['y_pred_raw'].rank(method='first', ascending=False)
    eval_df['rank_pct'] = (eval_df['runners'] - eval_df['rank']) / (eval_df['runners'] - 1)
    eval_df['winnings'] = eval_df.apply(lambda r: calculate_win_profit(r['position'], r['win_sp']), axis=1)

    _is_past = eval_df['date'] < today
    _is_top  = eval_df['rank_pct'] == 1
    eval_df['is_top_pick'] = (_is_past & _is_top).astype(int)

    _odds_ok = pd.Series(True, index=eval_df.index)
    if BET_ODDS_MIN is not None:
        _odds_ok &= eval_df['win_sp'] >= BET_ODDS_MIN
    if BET_ODDS_MAX is not None:
        _odds_ok &= eval_df['win_sp'] <= BET_ODDS_MAX

    _mrp_ok = pd.Series(True, index=eval_df.index)
    if BET_MRP_MIN is not None:
        _mrp_ok &= eval_df['market_rank_pct'] >= BET_MRP_MIN
    if BET_MRP_MAX is not None:
        _mrp_ok &= eval_df['market_rank_pct'] <= BET_MRP_MAX

    # ── is_bet: top pick + odds filter (original strategy) ───────────────────
    eval_df['is_bet'] = (_is_past & _is_top & _odds_ok & _mrp_ok).astype(int)

    # ── is_bet_ev: Benter EV strategy — highest positive EV per race ─────────
    # Per race: pick the horse with the highest EV > 0 (regardless of rank).
    # Mirrors the reference paper betting strategy exactly.
    _ev_pos    = _is_past & (eval_df['ev_raw'] > 0) & _odds_ok & _mrp_ok
    _ev_maxrank = eval_df.groupby('raceId')['ev_raw'].rank(method='first', ascending=False)
    eval_df['is_bet_ev'] = (_ev_pos & (_ev_maxrank == 1)).astype(int)

    return eval_df


def _odds_band(sp):
    if sp < 3:   return '<3'
    if sp <= 6:  return '3-6'
    return '>6'


def _print_group_stats(grp, label, stake=10):
    n = len(grp)
    if n == 0:
        print(f'  {label}: 0 bets')
        return
    wins_win   = (grp['win']   == 1).sum()
    wins_place = (grp['place'] == 1).sum()
    roi_win    = 100 * (grp[grp['win']   == 1]['win_sp'].sum()   / n - 1)
    roi_place  = 100 * (grp[grp['place'] == 1]['place_sp'].sum() / n - 1)
    mean_sp    = grp[grp['win'] == 1]['win_sp'].mean() if wins_win > 0 else float('nan')
    print(f'  {label}:')
    print(f'    Bets={n}  Wins={wins_win}  SR={100*wins_win/n:.1f}%  MeanSP={mean_sp:.2f}')
    print(f'    ROI win={roi_win:+.2f}%  ROI place={roi_place:+.2f}%')
    print(f'    Profit={stake*(roi_win/100)*n:+.1f}€  (win, {stake}€/bet)')


def print_betting_stats(eval_df, stake=10, label='Model'):
    top_all  = eval_df[eval_df['is_top_pick'] == 1]
    top_bet  = eval_df[eval_df['is_bet']      == 1]
    top_ev   = eval_df[eval_df['is_bet_ev']   == 1]

    print(f'\n══ {label} ════════════════════════════════')
    _print_group_stats(top_all, f'All top-picks (n={len(top_all)})', stake)

    if BET_ODDS_MIN is not None or BET_ODDS_MAX is not None or BET_MRP_MIN is not None or BET_MRP_MAX is not None:
        _lo  = BET_ODDS_MIN or '—'
        _hi  = BET_ODDS_MAX or '—'
        _mlo = f'  MRP≥{BET_MRP_MIN}' if BET_MRP_MIN is not None else ''
        _mhi = f'  MRP≤{BET_MRP_MAX}' if BET_MRP_MAX is not None else ''
        _print_group_stats(top_bet, f'Odds {_lo}–{_hi}{_mlo}{_mhi} filter (n={len(top_bet)})', stake)

    # ── EV strategy: top positive-EV horse per race ───────────────────────────
    _print_group_stats(top_ev, f'EV strategy — best EV>0 per race (n={len(top_ev)})', stake)

    print('\n  ROI by odds band (all top picks):')
    print(f'  {"Band":<8} {"Bets":>5} {"Wins":>5} {"SR%":>7} {"ROI%":>8}')
    print(f'  {"-"*40}')
    for band in ['<3', '3-6', '>6']:
        grp = top_all[top_all['win_sp'].apply(_odds_band) == band]
        n   = len(grp)
        if n == 0:
            continue
        w   = (grp['win'] == 1).sum()
        r   = 100 * (grp[grp['win'] == 1]['win_sp'].sum() / n - 1)
        marker = ' ◄ BET' if band == '3-6' and BET_ODDS_MIN == 3.0 and BET_ODDS_MAX == 6.0 else ''
        print(f'  {band:<8} {n:>5} {w:>5} {100*w/n:>6.1f}% {r:>7.2f}%{marker}')

    if len(top_all) > 0:
        _top_prob   = top_all['y_pred_raw'].mean()
        _field_prob = eval_df['y_pred_raw'].mean()
        print(f'\n  Mean prob — top picks: {_top_prob:.4f}  field: {_field_prob:.4f}  '
              f'lift: {_top_prob/_field_prob:.2f}×')


y_eval = build_evaluation_df(df, y, label_preds_prob, mode=MODE)

if MODE == 'train':
    _oos = y_eval[y_eval['date'] >= _cutoff]
    print(f'NOTE: Train mode — section 8 shows OOS test period only ({_cutoff.date()} onward).')
    print_betting_stats(_oos, stake=10, label='OOS test period')
else:
    print_betting_stats(y_eval, stake=10)

### 8.1  Diagnostic Breakdown — where does the edge come from?

In [ ]:
# ── Use the same OOS slice as section 8 ──────────────────────────────────────
_diag = y_eval.copy()
if MODE == 'train':
    _diag = _diag[_diag['date'] >= _cutoff]

_picks = _diag[_diag['is_top_pick'] == 1].copy()

if len(_picks) == 0:
    print('No top-picks available for diagnostic breakdown.')
else:
    # ── Model confidence: top prob / 2nd-highest prob within each race ────────
    _top2     = _diag.groupby('raceId')['y_pred_raw'].apply(
        lambda x: sorted(x.values, reverse=True)[:2]
    )
    _conf_map = _top2.apply(lambda p: p[0] / (p[1] + 1e-9) if len(p) > 1 else 1.0).to_dict()
    _picks['confidence'] = _picks['raceId'].map(_conf_map)

    def _row(grp, label, min_bets=10):
        n = len(grp)
        if n < min_bets: return
        w   = (grp['win'] == 1).sum()
        roi = 100 * (grp[grp['win'] == 1]['win_sp'].sum() / n - 1)
        marker = ' ◄' if roi > 2 else ''
        print(f'  {label:<24} {n:>5} {w:>5} {100*w/n:>6.1f}% {roi:>+7.2f}%{marker}')

    def _section(title):
        sep = '─' * max(0, 44 - len(title))
        print(f'\n  ── {title} {sep}')
        print(f'  {"Group":<24} {"Bets":>5} {"Wins":>5} {"SR%":>7} {"ROI%":>8}')
        print(f'  {"─"*50}')

    print('═' * 57)
    print('  8.1  DIAGNOSTIC BREAKDOWN  (top-picks only)')
    print('═' * 57)

    _section('FIELD SIZE')
    _picks['_fs'] = pd.cut(_picks['runners'], bins=[0, 7, 12, 99],
                           labels=['Small ≤7', 'Medium 8–12', 'Large >12'])
    for name, g in _picks.groupby('_fs', observed=True): _row(g, str(name))

    if 'racetype' in _picks.columns:
        _section('RACE TYPE')
        for name, g in _picks.groupby('racetype', observed=True): _row(g, str(name))

    if 'going_category' in _picks.columns:
        _section('GOING')
        for name, g in _picks.groupby('going_category', observed=True): _row(g, str(name))

    if 'distype' in _picks.columns:
        _section('DISTANCE GROUP')
        for name, g in _picks.groupby('distype', observed=True): _row(g, str(name))

    _section('MODEL CONFIDENCE (top ÷ 2nd prob in race)')
    try:
        _picks['_cq'] = pd.qcut(_picks['confidence'], q=4,
                                 labels=['Q1 low', 'Q2', 'Q3', 'Q4 high'], duplicates='drop')
        for name, g in _picks.groupby('_cq', observed=True): _row(g, str(name))
    except Exception:
        print('  (not enough data for confidence quartiles)')

    if 'market_rank_pct' in _picks.columns:
        _section('MODEL vs MARKET (market_rank_pct of top pick)')
        _picks['_mg'] = pd.cut(
            _picks['market_rank_pct'], bins=[-0.001, 0.25, 0.5, 0.75, 1.001],
            labels=['Fav zone (0–25%)', 'Near fav (25–50%)',
                    'Near out (50–75%)', 'Outsider (75–100%)']
        )
        for name, g in _picks.groupby('_mg', observed=True): _row(g, str(name))

    # ── EV strategy breakdown ─────────────────────────────────────────────────
    _ev_picks = _diag[_diag['is_bet_ev'] == 1].copy()
    if len(_ev_picks) >= 10:
        _section('EV STRATEGY — odds band breakdown')
        for band in ['<3', '3-6', '>6']:
            g = _ev_picks[_ev_picks['win_sp'].apply(_odds_band) == band]
            _row(g, band)

    # ── Calibration ───────────────────────────────────────────────────────────
    print(f'\n  ── CALIBRATION (all horses with results) ──────────')
    print(f'  {"Decile":<10} {"N":>6} {"Actual%":>9} {"Pred%":>8} {"Diff":>8}')
    print(f'  {"─"*45}')
    try:
        _diag['_dec'] = pd.qcut(_diag['y_pred_raw'], q=10, labels=False, duplicates='drop')
        for d, g in _diag.groupby('_dec'):
            act  = 100 * g['place'].mean()
            pred = 100 * g['y_pred_raw'].mean()
            diff = act - pred
            flag = ' ▲' if diff > 3 else (' ▼' if diff < -3 else '')
            print(f'  D{int(d)+1:<9} {len(g):>6} {act:>8.1f}% {pred:>7.1f}% {diff:>+7.1f}%{flag}')
    except Exception as e:
        print(f'  Error: {e}')

    # ── Benter two-step model summary ─────────────────────────────────────────
    if MODE == 'train' and blend_net is not None:
        _w = blend_net.w.weight.data.cpu().numpy().flatten()
        _alpha, _beta = float(_w[0]), float(_w[1])
        print(f'\n  ── BENTER TWO-STEP MODEL ───────────────────────────')
        print(f'  Step 1  LGBM clf      AUC(val)={_fund_best_val:.4f}  '
              f'AUC win={auc_fund_win:.4f}  place={auc_fund_plc:.4f}')
        print(f'  Market-only                       '
              f'AUC win={auc_mkt_win:.4f}  place={auc_mkt_plc:.4f}')
        print(f'  Step 2  blend clogit  best_val={_blend_best_val:.4f}  '
              f'AUC win={auc_blend_win:.4f}  place={auc_blend_plc:.4f}')
        print(f'  α (fund)={_alpha:.4f}  β (market)={_beta:.4f}  '
              f'β/α={_beta/_alpha:.3f}')
        if fund_clf is not None:
            n_trees = fund_clf.booster_.num_trees()
            print(f'  fund_clf trees: {n_trees}  best_iter: {fund_clf.best_iteration_}  '
                  f'|  blend_net params: 2')

## 9  Today's Tips

In [ ]:
today_tips = (
    y_eval[y_eval['date'] == TODAY]
    .sort_values('SP')
    [['raceId', 'date', 'course', 'racetype', 'horse', 'distype', 'position', 'win_sp', 'is_top_pick', 'SP']]
)

today_tips.to_parquet(PATHS['tips_out'], engine='pyarrow', index=False)
today_tips
